In [1]:
import pandas as pd
import numpy as np
import fits_convert_trackmate
import fits_convert_trackmate_alt
import fits_convert
import msd_calc
import calc_anisotropy
import calc_lifetime
import utils

In [ ]:
## Configure parameters

# Directories to use. Formatted as a dictionary with filename: condition. Graphs are ordered by condition's first appearance
# Since everything goes to Matplotlib, condition can be formatted with TEX to get superscripts/special chars
# Fun latex tip: use \\mathsf{} to get non-italic font
# Since names get clunky quickly, I assign the names to variables and use those for configuration


DATA_DIRECTORIES = {
    "/home/dl/work/lsa-jsbiteen/MIGRATED/Lab_Members/Sam_Steen/Data/260115_PAmCherry-Swi6_wt_20ms": "wt",
    "/home/dl/work/lsa-jsbiteen/MIGRATED/Lab_Members/Sam_Steen/Data/260116_PAmCherry-Swi6_wt-and-sj_20ms/wt": "wt",
    "/home/dl/work/lsa-jsbiteen/MIGRATED/Lab_Members/Sam_Steen/Data/260911_PAmCherry-Swi6_varied_20ms/non-neg": "non-neg",
    "/home/dl/work/lsa-jsbiteen/MIGRATED/Lab_Members/Sam_Steen/Data/260912_PAmCherry-Swi6_varied_20ms/non-neg": "non-neg",
}

EXCLUDED_FILES = []

# Settings for MSD
MSD_SETTINGS = {'t_int': .02, # Integration time in seconds
                't_delay': 0,  # Define the time delay between frames in seconds
                'min_frames': 4, # Define minimum track length (in frames)
                'max_gap': 2, # Define maximum frame allowed within a track
                'pixel_size_um': 0.049, # Define the pixel size in um
                }

# TODO: maybe merge n_components and criterium?
# Settings for gaussian mixture model fitting (to MSD)
GMM_SETTINGS = {'n_init': 1, # Number of iterations to run
                'n_components': 2, # Number of curves to fit to. Can be an integer or 
                                   # a dictionary (condition: number) or the string "optimize" to find an ideal value.
                'criterium': 'bic', # Criterium for selecting the best model. Can be aic or bic.
                'bootstrap_n': 100, # Number of times to bootstrap data for CI. By far slower than anisotropy stuff-- 100 max
                'verbose': True
                }


# Settings for Anisotropy
ANISOTROPY_SETTINGS = {'min_D': .1, # um^2/sec. Minimum distance to be considered
                       'max_D': 100, # um^2/sec. Maximum distance to be considered
                       'rose_n_bins': 16, # Number of bins to use on the roseplots
                       'angle_from_center': 30, # Degrees in each direction from 0/180° for calculating fold anisotropy
                       'pixel_size_um': MSD_SETTINGS['pixel_size_um'], # Adjust in MSD_SETTINGS
                       'usable_range': (0, .35), # mean displacements to consider for the fold anisotropy graphs
                       'disp_n_bins': 10, # How many mean displacement bins to use on the fold anisotropy graphs
                       'bootstrap_n': 100, # How many bootstrap samples to collect (for error bars on graphs). Often 100-10000.
                       'permutation_n': 100,  # How many permutation samples to collect (for p-values). Often 1000-100000.
                       'min_mean_displacement': .035, # For doing the bar chart comparison only on anisotropies (cont'd next line)
                       'max_mean_displacement': .175 # (cont'd) calculated from steps with a certain range of mean displacements
                      }

In [ ]:
# Initialize combined_tracks (1 row per track) by reading in data from DATA_DIRECTORIES
combined_tracks = fits_convert_trackmate.batch_convert_all_folders(DATA_DIRECTORIES, EXCLUDED_FILES)

# Initialize combined_info (1 row per condition) as a blank data frame
combined_info = pd.DataFrame(data={'condition': list(dict.fromkeys(DATA_DIRECTORIES.values()))}).set_index('condition')

# Initialize general_info for non-condition-specific info. By default, stores all variables in all caps (constants)
general_info = {k: v for k, v in globals().items() if k.isupper() and not k.startswith("__")}

# Save all to file. This is done automatically elsewhere, but manually here since we just created some structures
utils.pickle_save(ct=combined_tracks, ci=combined_info, gi=general_info)

In [4]:
combined_tracks = msd_calc.calc_squared_displacement(combined_tracks, MSD_SETTINGS)

In [5]:
## This cell: calculate MSDs
# Calculate MSD for each track (uses Chris' code, see other file for details)
# combined_tracks, general_info = calc_msd(combined_tracks, general_info, MSD_SETTINGS)
# combined_tracks, general_info = msd_chris(combined_tracks, general_info, MSD_SETTINGS)
combined_tracks, general_info = msd_calc.get_msd(combined_tracks, general_info, MSD_SETTINGS)

In [6]:
# Calculate GMMs and store them as columns (n_comps, means, variances, weights) in combined_info
combined_info = msd_calc.calc_gmms(combined_tracks, combined_info, GMM_SETTINGS)

wt
non-neg
non-pos_lowac
swap


In [7]:
# Calculate angle (for anisotropy) for each track
combined_tracks = calc_anisotropy.calc_angle(combined_tracks, False, ANISOTROPY_SETTINGS)
# Calculate fold anisotropy for each condition
combined_info = calc_anisotropy.calc_fold_anisotropy(combined_info, combined_tracks, ANISOTROPY_SETTINGS)

/home/dl/work/python/python-all-the-way-down/calc_anisotropy.py:96: RuntimeWarning: invalid value encountered in scalar divide
  angle = np.degrees(np.arccos(np.dot(A, B)/((A[0]**2 + A[1]**2)**.5 * (B[0]**2 + B[1]**2)**.5)))
/home/dl/work/python/python-all-the-way-down/calc_anisotropy.py:96: RuntimeWarning: invalid value encountered in arccos
  angle = np.degrees(np.arccos(np.dot(A, B)/((A[0]**2 + A[1]**2)**.5 * (B[0]**2 + B[1]**2)**.5)))


In [8]:
combined_info = calc_anisotropy.get_anisotropy_by_displacement(combined_tracks, combined_info, ANISOTROPY_SETTINGS)

In [10]:
combined_info = calc_anisotropy.summarize_midrange_anisotropy(combined_tracks, combined_info, ANISOTROPY_SETTINGS)

wt vs wt p-val: 1.0
wt vs non-neg p-val: 0.0
wt vs non-pos_lowac p-val: 0.12
wt vs swap p-val: 0.03
